# GAP-ITSM: ITSM + Overnight Gap Confirmation + Vol Filter
## CSCV/PBO Overfitting Analysis

Regime-invariant improvements vs ITSM-ATR (PBO=60.6%):
- Gap confirm: BINARY filter, no continuous threshold
- Vol filter: high-ATR days only (ITSM stronger in high-vol, Baltussen et al. 2021)
- Fixed 2:1 RR: removes TP as free parameter
- CSCV grid: 2 regime-invariant dims (vol_lookback x sl_atr_mult), N=9

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
sys.path.insert(0, os.path.abspath('.'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
INITIAL_EQUITY = 50_000.0

In [2]:
csv = os.path.join(ROOT, 'data', 'NQ_5m.csv')
df = pd.read_csv(csv, index_col=0)
df.index = pd.to_datetime(df.index, utc=True).tz_convert('America/New_York')
df.columns = [c.lower() for c in df.columns]
print(f'Bars loaded : {len(df):,}')
print(f'Date range  : {df.index[0].date()} to {df.index[-1].date()}')

Bars loaded : 222,295
Date range  : 2014-12-19 to 2026-03-17


In [3]:
from strategy_itsm_atr import run_backtest as itsm_atr_bt
from strategy_gap_itsm import run_backtest as gap_bt

def perf(name, result):
    trades, eq = result
    if trades.empty: return None
    dp = trades.groupby('date')['pnl'].sum()
    sr = dp.mean() / dp.std() * np.sqrt(252)
    peak = eq.cummax(); dd = ((eq - peak) / peak).min()
    wr = (trades['pnl'] > 0).mean()
    tr = (trades['equity_after'].iloc[-1] - INITIAL_EQUITY) / INITIAL_EQUITY * 100
    sl_pct = (trades['exit_reason']=='SL').mean()
    tp_pct = (trades['exit_reason']=='TP').mean()
    tm_pct = trades['exit_reason'].isin(['TIME','EOD']).mean()
    return {'name': name, 'trades': len(trades), 'wr': wr, 'sharpe': sr,
            'dd': dd, 'tr': tr, 'sl': sl_pct, 'tp': tp_pct, 'time': tm_pct, 'eq': eq}

configs = [
    ('V1: ITSM-ATR (no filters)', itsm_atr_bt(df, sl_atr_mult=0.5, tp_atr_mult=1.0, itsm_threshold=0.002, itsm_bars=6)),
    ('V2: + Gap Confirm only',    gap_bt(df, sl_atr_mult=0.5, require_gap_confirm=True, require_high_vol=False)),
    ('V3: + Vol Filter only',     gap_bt(df, sl_atr_mult=0.5, require_gap_confirm=False, require_high_vol=True)),
    ('V4: GAP-ITSM (both)',       gap_bt(df, sl_atr_mult=0.5)),
]
results = [perf(n, r) for n, r in configs]

hdr = f'{"Strategy":<35} {"Trades":>7} {"WinRate":>8} {"Sharpe":>8} {"MaxDD":>8} {"TotalRet":>10}'
print(hdr)
for r in results:
    if r: print(f'{r["name"]:<35} {r["trades"]:>7} {r["wr"]:>8.1%} {r["sharpe"]:>8.3f} {r["dd"]:>8.1%} {r["tr"]:>9.1f}%')
print(f'  S&P 500 benchmark              -        -    0.600  -55.0%     170.0%')

Strategy                             Trades  WinRate   Sharpe    MaxDD   TotalRet
V1: ITSM-ATR (no filters)              1620    49.0%    1.078   -17.8%      95.7%
V2: + Gap Confirm only                  808    51.0%    1.574    -9.3%      70.9%
V3: + Vol Filter only                   905    48.7%    0.852   -24.7%      41.3%
V4: GAP-ITSM (both)                     457    54.0%    1.987   -13.3%      49.0%
  S&P 500 benchmark              -        -    0.600  -55.0%     170.0%


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3','#4CAF50','#FF9800','#F44336']
labels = ['V1: ITSM-ATR','V2: +Gap','V3: +VolFilter','V4: GAP-ITSM']
ax = axes[0]
for i, r in enumerate(results):
    if r and r['eq'] is not None:
        eq = r['eq']
        ax.plot(eq.index, (eq / INITIAL_EQUITY - 1) * 100, label=labels[i], color=colors[i], linewidth=1.5)
try:
    import yfinance as yf
    spy = yf.download('SPY', start='2015-01-01', end='2026-03-17', auto_adjust=True, progress=False)
    spy_r = (spy['Close'] / spy['Close'].iloc[0] - 1) * 100
    ax.plot(spy_r.index, spy_r, 'k--', label='S&P 500', linewidth=1.0, alpha=0.6)
except: pass
ax.set_title('Equity Curves: Filter Progression'); ax.set_ylabel('Return (%)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax2 = axes[1]
sharpes = [r['sharpe'] for r in results if r]
dds = [-r['dd']*100 for r in results if r]
x = np.arange(len(sharpes))
ax2.bar(x - 0.2, sharpes, 0.35, label='Sharpe', color=colors[:len(sharpes)])
ax2.bar(x + 0.2, dds, 0.35, label='Max DD %', color=colors[:len(sharpes)], alpha=0.5, hatch='/')
ax2.axhline(1.0, color='green', linestyle='--', alpha=0.8, label='Sharpe target')
ax2.axhline(10.0, color='red', linestyle='--', alpha=0.8, label='DD target')
ax2.set_xticks(x); ax2.set_xticklabels(['V1\nITSM-ATR','V2\n+Gap','V3\n+Vol','V4\nBoth'], fontsize=8)
ax2.set_title('Sharpe vs Max DD'); ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'output', 'gap_itsm_backtest.png'), dpi=120, bbox_inches='tight')
plt.show(); print('Saved gap_itsm_backtest.png')

Saved gap_itsm_backtest.png


In [5]:
from cscv_gap_itsm import build_pnl_matrix, run_cscv, pbo_verdict

SL_MULTS = [0.5, 0.75, 1.0]
VOL_LBS  = [42, 63, 126]
variants = [{'sl_atr_mult': sl, 'vol_lookback': vl}
            for sl in SL_MULTS for vl in VOL_LBS]
N = len(variants)
print(f'N = {N} variants (regime-invariant grid: sl_atr_mult x vol_lookback)')
for v in variants:
    print(f'  SL={v["sl_atr_mult"]}xATR  vol_lookback={v["vol_lookback"]}d')

N = 9 variants (regime-invariant grid: sl_atr_mult x vol_lookback)
  SL=0.5xATR  vol_lookback=42d
  SL=0.5xATR  vol_lookback=63d
  SL=0.5xATR  vol_lookback=126d
  SL=0.75xATR  vol_lookback=42d
  SL=0.75xATR  vol_lookback=63d
  SL=0.75xATR  vol_lookback=126d
  SL=1.0xATR  vol_lookback=42d
  SL=1.0xATR  vol_lookback=63d
  SL=1.0xATR  vol_lookback=126d


In [6]:
print(f'Building PnL matrix ({N} variants)...')
M, dates = build_pnl_matrix(df, variants)
print(f'Shape: {M.shape}  (T={M.shape[0]} days, N={M.shape[1]} variants)')
print(f'Range: {dates[0].date()} to {dates[-1].date()}')

Building PnL matrix (9 variants)...


  3/9 variants complete...


  6/9 variants complete...


  9/9 variants complete...
Shape: (551, 9)  (T=551 days, N=9 variants)
Range: 2014-12-24 to 2026-03-16


In [7]:
print('Running CSCV  S=16  C(16,8)=12,870 combinations...')
res = run_cscv(M, S=16)
pbo = res['pbo']
print()
print(f'PBO  = {pbo:.1%}  {pbo_verdict(pbo)}')
print(f'Median IS Sharpe  : {np.median(res["is_sharpes"]):.3f}')
print(f'Median OOS Sharpe : {np.median(res["oos_sharpes"]):.3f}')
deg = (np.median(res['is_sharpes']) - np.median(res['oos_sharpes'])) / max(np.median(res['is_sharpes']), 1e-6) * 100
print(f'Sharpe degradation: {deg:.1f}%')
print(f'Prob(loss OOS)    : {np.mean(res["oos_sharpes"] < 0):.1%}')
print()
print('Comparison with previous strategies:')
print(f'  ORB v4 (fixed SL/TP)     : 47.7%  MODERATE')
print(f'  ITSM-ATR (abs threshold) : 60.6%  HIGH')
print(f'  GAP-ITSM (binary filter) : {pbo:.1%}  {pbo_verdict(pbo)}')

Running CSCV  S=16  C(16,8)=12,870 combinations...



PBO  = 58.2%  HIGH OVERFITTING RISK (PBO >= 50%)
Median IS Sharpe  : 2.082
Median OOS Sharpe : 1.325
Sharpe degradation: 36.4%
Prob(loss OOS)    : 1.7%

Comparison with previous strategies:
  ORB v4 (fixed SL/TP)     : 47.7%  MODERATE
  ITSM-ATR (abs threshold) : 60.6%  HIGH
  GAP-ITSM (binary filter) : 58.2%  HIGH OVERFITTING RISK (PBO >= 50%)


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
logits = res['logits']
ax.hist(logits, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='red', linewidth=1.5, linestyle='--', label='PBO boundary')
ax.set_title(f'Logit Distribution  PBO={np.mean(logits<0)*100:.1f}%')
ax.set_xlabel('Logit'); ax.set_ylabel('Frequency')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
is_sr = res['is_sharpes']; oos_sr = res['oos_sharpes']
ax.scatter(is_sr, oos_sr, alpha=0.1, s=4, color='steelblue')
vmin = min(is_sr.min(), oos_sr.min()); vmax = max(is_sr.max(), oos_sr.max())
ax.plot([vmin, vmax], [vmin, vmax], 'k--', linewidth=1, label='IS=OOS (ideal)')
ax.axhline(0, color='red', linewidth=0.8, linestyle=':')
ax.set_xlabel('IS Sharpe'); ax.set_ylabel('OOS Sharpe')
ax.set_title('IS vs OOS Sharpe'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[2]
pbo_sl, pbo_vl = [], []
for sl_val in SL_MULTS:
    idxs = [i for i, v in enumerate(variants) if v['sl_atr_mult'] == sl_val]
    r2 = run_cscv(M[:, idxs], S=16)
    pbo_sl.append(r2['pbo'] * 100)
for vl_val in VOL_LBS:
    idxs = [i for i, v in enumerate(variants) if v['vol_lookback'] == vl_val]
    r2 = run_cscv(M[:, idxs], S=16)
    pbo_vl.append(r2['pbo'] * 100)
x = np.arange(3)
ax.bar(x - 0.2, pbo_sl, 0.35, label='SL sub-grid', color='#2196F3', alpha=0.8)
ax.bar(x + 0.2, pbo_vl, 0.35, label='vol_lb sub-grid', color='#FF9800', alpha=0.8)
ax.axhline(20, color='green', linestyle='--', alpha=0.8, label='20% target')
ax.axhline(50, color='red', linestyle='--', alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(['0.5/42d','0.75/63d','1.0/126d'], fontsize=8)
ax.set_title('PBO by Parameter Dimension')
ax.set_ylabel('PBO (%)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'output', 'gap_itsm_cscv.png'), dpi=120, bbox_inches='tight')
plt.show(); print('Saved gap_itsm_cscv.png')

Saved gap_itsm_cscv.png


In [9]:
print('Per-variant full-dataset performance:')
print(f'{"Variant":<30} {"Trades":>7} {"Sharpe":>8} {"MaxDD":>8} {"Sharpe>1":>9} {"DD<10%":>8}')
pass_both = 0
for v in variants:
    t, eq = gap_bt(df, **v)
    if t.empty: continue
    dp = t.groupby('date')['pnl'].sum()
    sr = dp.mean() / dp.std() * np.sqrt(252)
    peak = eq.cummax(); dd = ((eq - peak) / peak).min()
    ps = 'PASS' if sr > 1.0 else 'FAIL'
    pd_ = 'PASS' if dd > -0.10 else 'FAIL'
    if ps == 'PASS' and pd_ == 'PASS': pass_both += 1
    name = f"SL={v['sl_atr_mult']}  vol={v['vol_lookback']}d"
    print(f'{name:<30} {len(t):>7} {sr:>8.3f} {dd:>8.1%} {ps:>9} {pd_:>8}')
print(f'\nVariants passing BOTH targets: {pass_both}/{len(variants)}')
print()
print('=== FINAL VERDICT ===')
print(f'PBO = {res["pbo"]:.1%}  --  {pbo_verdict(res["pbo"])}')
print(f'Prob(loss OOS)   : {np.mean(res["oos_sharpes"] < 0):.1%}')
print(f'Median OOS Sharpe: {np.median(res["oos_sharpes"]):.3f}')
print(f'Variants passing Sharpe>1.0 AND DD<10%: {pass_both}/{len(variants)}')

Per-variant full-dataset performance:
Variant                         Trades   Sharpe    MaxDD  Sharpe>1   DD<10%


SL=0.5  vol=42d                    428    1.973    -9.6%      PASS     PASS


SL=0.5  vol=63d                    457    1.987   -13.3%      PASS     FAIL


SL=0.5  vol=126d                   460    1.498    -8.3%      PASS     PASS


SL=0.75  vol=42d                   428    1.495    -7.6%      PASS     PASS


SL=0.75  vol=63d                   457    1.450   -10.9%      PASS     FAIL


SL=0.75  vol=126d                  460    0.852   -11.2%      FAIL     FAIL


SL=1.0  vol=42d                    428    1.779    -6.5%      PASS     PASS


SL=1.0  vol=63d                    457    1.684    -9.0%      PASS     PASS


SL=1.0  vol=126d                   460    1.411    -7.9%      PASS     PASS

Variants passing BOTH targets: 6/9

=== FINAL VERDICT ===
PBO = 58.2%  --  HIGH OVERFITTING RISK (PBO >= 50%)
Prob(loss OOS)   : 1.7%
Median OOS Sharpe: 1.325
Variants passing Sharpe>1.0 AND DD<10%: 6/9
